# Phase 3: Alpha Research & Statistical Learning  
## Notebook 03_03 — Markov Regime Switching

### Research question

Can hidden Markov models identify latent market regimes from returns and volatility features, and can those regimes be used for risk control, volatility forecasting, and alpha research?

This notebook follows naturally from:

```text
03_01_garch_volatility_modeling
03_02_har_realized_volatility_forecasting
```

GARCH and HAR-RV model volatility as a continuously evolving process. Markov regime switching instead assumes that the market moves between **discrete hidden states**:

- calm / risk-on;
- normal / neutral;
- volatile / risk-off;
- crash / stress.

This notebook covers:

1. hidden Markov model intuition;
2. Markov transition matrices;
3. synthetic regime-switching return data;
4. Gaussian emissions;
5. log-space forward algorithm;
6. backward algorithm;
7. Baum-Welch / EM estimation;
8. Viterbi most-likely regime path;
9. posterior regime probabilities;
10. regime statistics and interpretation;
11. transition matrix diagnostics;
12. out-of-sample regime filtering;
13. regime-conditioned risk forecasts;
14. a toy regime-filtered strategy;
15. limitations and research-frontier extensions.

The key idea:

> A regime model treats market behaviour as state-dependent: the return distribution changes depending on an unobserved market state.

## 1. Why regime switching?

Financial markets often behave differently across periods:

| Regime | Typical behaviour |
|---|---|
| Calm | low volatility, stable trends |
| Normal | moderate volatility |
| Stress | high volatility, negative skew |
| Crisis | extreme volatility, large drawdowns |

A single unconditional model can blur these together.

A hidden Markov model assumes a latent state:

$$
Z_t \in \{1,2,\dots,K\}
$$

and observations:

$$
X_t
$$

The hidden state follows a Markov chain:

$$
\mathbb P(Z_t=j \mid Z_{t-1}=i)=A_{ij}
$$

where $A$ is the transition matrix.

The observation distribution depends on the hidden state:

$$
X_t \mid Z_t=k \sim \mathcal N(\mu_k,\Sigma_k)
$$

The model infers regimes from observed data.

## 2. The HMM likelihood problem

For observations:

$$
X_{1:T}
$$

and hidden states:

$$
Z_{1:T}
$$

the joint likelihood is:

$$
p(X_{1:T},Z_{1:T}) = \pi_{Z_1} p(X_1\mid Z_1) \prod_{t=2}^{T} A_{Z_{t-1},Z_t} p(X_t\mid Z_t)
$$

where:

- $\pi$ is the initial state probability;
- $A$ is the transition matrix;
- $p(X_t\mid Z_t)$ is the emission density.

Directly summing over all possible hidden paths is impossible for long series because there are:

$$
K^T
$$

paths.

The forward algorithm solves this recursively in:

$$
O(TK^2)
$$

## 3. Why log-space?

Forward probabilities can become extremely small.

To avoid numerical underflow, we use log probabilities and the log-sum-exp identity:

$$
\log\sum_i e^{a_i} = m+\log\sum_i e^{a_i-m}
$$

where:

$$
m=\max_i a_i
$$

This is essential for stable HMM inference.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class RegimeConfig:
    n_obs: int = 2_500
    n_states: int = 3
    annualisation: int = 252
    train_fraction: float = 0.70
    seed: int = 42


config = RegimeConfig()
config

## 5. Synthetic regime-switching market data

We simulate a 3-regime market:

| Regime | Interpretation | Mean | Volatility |
|---:|---|---:|---:|
| 0 | calm/risk-on | positive | low |
| 1 | neutral/choppy | small | medium |
| 2 | stress/risk-off | negative | high |

The hidden regime follows a Markov transition matrix:

$$
A_{ij} = \mathbb P(Z_t=j \mid Z_{t-1}=i)
$$

High diagonal entries create persistence.

In [ ]:
def simulate_regime_switching_returns(config: RegimeConfig) -> pd.DataFrame:
    rng = np.random.default_rng(config.seed)

    n = config.n_obs
    K = config.n_states

    transition = np.array([
        [0.965, 0.030, 0.005],
        [0.040, 0.920, 0.040],
        [0.010, 0.080, 0.910],
    ])

    means = np.array([0.00055, 0.00005, -0.00085])
    vols = np.array([0.0060, 0.0110, 0.0240])

    states = np.zeros(n, dtype=int)
    returns = np.zeros(n)

    states[0] = 0

    for t in range(1, n):
        states[t] = rng.choice(K, p=transition[states[t - 1]])
        returns[t] = means[states[t]] + vols[states[t]] * rng.standard_t(df=7) * np.sqrt((7 - 2) / 7)

    dates = pd.bdate_range("2015-01-01", periods=n)

    out = pd.DataFrame({
        "timestamp": dates,
        "true_state": states,
        "return": returns
    })

    out["price"] = 100 * np.exp(out["return"].cumsum())
    out["squared_return"] = out["return"] ** 2
    out["abs_return"] = out["return"].abs()
    out["rolling_vol_21d"] = out["return"].rolling(21).std() * np.sqrt(config.annualisation)
    out["rolling_vol_63d"] = out["return"].rolling(63).std() * np.sqrt(config.annualisation)

    return out, transition, means, vols


market_df, true_transition, true_means, true_vols = simulate_regime_switching_returns(config)

market_df.head()

In [ ]:
pd.DataFrame(true_transition, columns=[f"to_{i}" for i in range(config.n_states)], index=[f"from_{i}" for i in range(config.n_states)])

## 6. Visualising hidden regimes

In real data, regimes are hidden.

In this synthetic notebook, we keep the true state for validation.

This lets us judge whether the HMM inference is working.

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(market_df["timestamp"], market_df["price"])
plt.title("Synthetic Regime-Switching Price Series")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(market_df["timestamp"], market_df["return"])
plt.title("Synthetic Returns")
plt.xlabel("Date")
plt.ylabel("Return")
plt.show()

plt.figure(figsize=(12, 3))
plt.plot(market_df["timestamp"], market_df["true_state"], drawstyle="steps-post")
plt.title("True Hidden Regime")
plt.xlabel("Date")
plt.ylabel("Regime")
plt.yticks([0, 1, 2])
plt.show()

## 7. Feature design

The simplest HMM uses returns only:

$$
X_t = r_t
$$

A more informative HMM can use multiple features:

$$
X_t =
\begin{bmatrix}
r_t \\
|r_t| \\
r_t^2
\end{bmatrix}
$$

This notebook uses a multivariate Gaussian HMM with features:

1. return;
2. absolute return;
3. 21-day rolling volatility.

The rolling volatility feature is lagged to avoid leakage.

In [ ]:
def build_hmm_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["rolling_vol_21d_lag1"] = out["rolling_vol_21d"].shift(1)
    out["abs_return"] = out["return"].abs()

    feature_cols = ["return", "abs_return", "rolling_vol_21d_lag1"]

    out = out.dropna(subset=feature_cols).reset_index(drop=True)

    # Scale features for numerical stability.
    for col in feature_cols:
        mean = out[col].mean()
        std = out[col].std(ddof=1)
        out[f"{col}_z"] = (out[col] - mean) / std

    z_cols = [f"{c}_z" for c in feature_cols]

    return out, feature_cols, z_cols


hmm_df, raw_feature_cols, feature_cols = build_hmm_features(market_df)

hmm_df[["timestamp", "true_state"] + raw_feature_cols + feature_cols].head()

## 8. Train/test split

We fit the HMM on the training period and filter regimes on the test period.

No shuffling is used.

In [ ]:
split_idx = int(len(hmm_df) * config.train_fraction)

train_df = hmm_df.iloc[:split_idx].copy()
test_df = hmm_df.iloc[split_idx:].copy()

X_train = train_df[feature_cols].to_numpy()
X_test = test_df[feature_cols].to_numpy()

pd.Series({
    "n_total": len(hmm_df),
    "n_train": len(train_df),
    "n_test": len(test_df),
    "train_start": train_df["timestamp"].iloc[0],
    "train_end": train_df["timestamp"].iloc[-1],
    "test_start": test_df["timestamp"].iloc[0],
    "test_end": test_df["timestamp"].iloc[-1]
})

## 9. Numerical helpers

We implement:

1. log-sum-exp;
2. multivariate Gaussian log density;
3. row normalisation for transition matrices;
4. stable covariance regularisation.

In [ ]:
def logsumexp(a, axis=None, keepdims=False):
    a = np.asarray(a, dtype=float)
    m = np.max(a, axis=axis, keepdims=True)
    out = m + np.log(np.sum(np.exp(a - m), axis=axis, keepdims=True))
    if not keepdims:
        out = np.squeeze(out, axis=axis)
    return out


def normalize_rows(matrix, floor=1e-12):
    mat = np.asarray(matrix, dtype=float)
    mat = np.maximum(mat, floor)
    return mat / mat.sum(axis=1, keepdims=True)


def gaussian_logpdf_full(X, means, covariances, min_covar=1e-6):
    """
    Multivariate Gaussian log density for each observation and state.

    Returns shape (T, K).
    """
    X = np.asarray(X, dtype=float)
    T, d = X.shape
    K = means.shape[0]

    log_probs = np.zeros((T, K))

    for k in range(K):
        cov = covariances[k] + min_covar * np.eye(d)
        inv_cov = np.linalg.pinv(cov)
        sign, logdet = np.linalg.slogdet(cov)

        if sign <= 0:
            cov = cov + 10 * min_covar * np.eye(d)
            inv_cov = np.linalg.pinv(cov)
            sign, logdet = np.linalg.slogdet(cov)

        diff = X - means[k]
        quad = np.sum(diff @ inv_cov * diff, axis=1)

        log_probs[:, k] = -0.5 * (d * np.log(2 * np.pi) + logdet + quad)

    return log_probs

## 10. Forward-backward algorithm

The forward recursion is:

$$
\alpha_t(j) = p(X_{1:t},Z_t=j)
$$

In log-space:

$$
\begin{aligned}
\log\alpha_t(j) &= \log p(X_t\mid Z_t=j) \\
&\quad + \log\sum_i \exp [ \log\alpha_{t-1}(i)+\log A_{ij} ]
\end{aligned}
$$

The backward recursion is:

$$
\beta_t(i) = p(X_{t+1:T}\mid Z_t=i)
$$

Posterior state probabilities are:

$$
\gamma_t(k) = p(Z_t=k\mid X_{1:T})
$$

In [ ]:
def forward_log(log_emissions, log_pi, log_A):
    T, K = log_emissions.shape
    log_alpha = np.zeros((T, K))

    log_alpha[0] = log_pi + log_emissions[0]

    for t in range(1, T):
        log_alpha[t] = log_emissions[t] + logsumexp(
            log_alpha[t - 1][:, None] + log_A,
            axis=0
        )

    log_likelihood = float(logsumexp(log_alpha[-1], axis=0))

    return log_alpha, log_likelihood


def backward_log(log_emissions, log_A):
    T, K = log_emissions.shape
    log_beta = np.zeros((T, K))

    for t in range(T - 2, -1, -1):
        log_beta[t] = logsumexp(
            log_A + log_emissions[t + 1][None, :] + log_beta[t + 1][None, :],
            axis=1
        )

    return log_beta


def forward_backward(log_emissions, pi, A):
    log_pi = np.log(np.maximum(pi, 1e-300))
    log_A = np.log(np.maximum(A, 1e-300))

    log_alpha, log_likelihood = forward_log(log_emissions, log_pi, log_A)
    log_beta = backward_log(log_emissions, log_A)

    log_gamma = log_alpha + log_beta - log_likelihood
    gamma = np.exp(log_gamma)

    T, K = log_emissions.shape
    xi = np.zeros((T - 1, K, K))

    for t in range(T - 1):
        log_xi_t = (
            log_alpha[t][:, None]
            + log_A
            + log_emissions[t + 1][None, :]
            + log_beta[t + 1][None, :]
            - log_likelihood
        )
        xi[t] = np.exp(log_xi_t)

    return {
        "log_alpha": log_alpha,
        "log_beta": log_beta,
        "gamma": gamma,
        "xi": xi,
        "log_likelihood": log_likelihood
    }

## 11. Initialising the HMM

HMM likelihoods are non-convex.

Initialisation matters.

We use quantiles of rolling volatility to initialise states:

- low-vol;
- medium-vol;
- high-vol.

This makes the estimated states easier to interpret.

In [ ]:
def initialise_hmm_from_volatility(train_df: pd.DataFrame, feature_cols: list[str], K: int):
    X = train_df[feature_cols].to_numpy()

    # Use raw lagged rolling volatility for intuitive state initialisation.
    vol = train_df["rolling_vol_21d_lag1"].to_numpy()
    quantiles = pd.qcut(vol, q=K, labels=False, duplicates="drop")

    means = []
    covariances = []

    for k in range(K):
        mask = quantiles == k

        if mask.sum() < X.shape[1] + 2:
            mask = np.ones(len(X), dtype=bool)

        means.append(X[mask].mean(axis=0))
        covariances.append(np.cov(X[mask].T) + 1e-4 * np.eye(X.shape[1]))

    means = np.array(means)
    covariances = np.array(covariances)

    pi = np.ones(K) / K

    # Persistent transition prior.
    A = np.full((K, K), 0.05 / (K - 1))
    np.fill_diagonal(A, 0.95)
    A = normalize_rows(A)

    return pi, A, means, covariances


pi0, A0, means0, covs0 = initialise_hmm_from_volatility(train_df, feature_cols, config.n_states)

pd.DataFrame(means0, columns=feature_cols)

## 12. Baum-Welch / EM estimation

The Baum-Welch algorithm is EM for HMMs.

### E-step

Compute:

$$
\gamma_t(k)=p(Z_t=k\mid X_{1:T})
$$

and:

$$
\xi_t(i,j)=p(Z_t=i,Z_{t+1}=j\mid X_{1:T})
$$

### M-step

Update:

$$
\pi_k=\gamma_1(k)
$$

$$
A_{ij} = \frac{\sum_{t=1}^{T-1}\xi_t(i,j)} {\sum_{t=1}^{T-1}\gamma_t(i)}
$$

Gaussian means and covariances are weighted by $\gamma_t(k)$.

In [ ]:
def fit_gaussian_hmm(
    X,
    K,
    pi_init,
    A_init,
    means_init,
    covs_init,
    max_iter=100,
    tol=1e-5,
    min_covar=1e-5
):
    X = np.asarray(X, dtype=float)
    T, d = X.shape

    pi = pi_init.copy()
    A = A_init.copy()
    means = means_init.copy()
    covariances = covs_init.copy()

    loglik_history = []

    for iteration in range(max_iter):
        log_emissions = gaussian_logpdf_full(X, means, covariances, min_covar=min_covar)
        fb = forward_backward(log_emissions, pi, A)

        gamma = fb["gamma"]
        xi = fb["xi"]
        loglik = fb["log_likelihood"]

        loglik_history.append(loglik)

        # M-step.
        pi = gamma[0] / gamma[0].sum()

        A = xi.sum(axis=0) / np.maximum(gamma[:-1].sum(axis=0)[:, None], 1e-12)
        A = normalize_rows(A)

        weights = gamma.sum(axis=0)

        for k in range(K):
            w = gamma[:, k]
            means[k] = (w[:, None] * X).sum(axis=0) / max(weights[k], 1e-12)

            diff = X - means[k]
            cov = (w[:, None] * diff).T @ diff / max(weights[k], 1e-12)
            covariances[k] = cov + min_covar * np.eye(d)

        if iteration > 0 and abs(loglik_history[-1] - loglik_history[-2]) < tol:
            break

    # Final posterior.
    log_emissions = gaussian_logpdf_full(X, means, covariances, min_covar=min_covar)
    fb = forward_backward(log_emissions, pi, A)

    return {
        "pi": pi,
        "A": A,
        "means": means,
        "covariances": covariances,
        "loglik_history": loglik_history,
        "posterior": fb["gamma"],
        "log_likelihood": fb["log_likelihood"],
        "n_iter": len(loglik_history)
    }


hmm_fit = fit_gaussian_hmm(
    X_train,
    K=config.n_states,
    pi_init=pi0,
    A_init=A0,
    means_init=means0,
    covs_init=covs0,
    max_iter=150,
    tol=1e-5
)

pd.Series({
    "log_likelihood": hmm_fit["log_likelihood"],
    "n_iter": hmm_fit["n_iter"]
})

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(hmm_fit["loglik_history"], marker="o")
plt.title("HMM EM Log-Likelihood")
plt.xlabel("Iteration")
plt.ylabel("Log-likelihood")
plt.show()

## 13. Sorting regimes by volatility

HMM state labels are arbitrary.

State 0 in the estimated model does not necessarily correspond to true state 0.

To make regimes interpretable, we sort estimated states by their annualised return volatility:

$$
\sigma_k = \operatorname{std}(r_t \mid Z_t=k)
$$

using posterior probabilities.

In [ ]:
def weighted_state_return_stats(df, posterior, K):
    rows = []
    r = df["return"].to_numpy()

    for k in range(K):
        w = posterior[:, k]
        w_sum = w.sum()
        mean = float((w * r).sum() / max(w_sum, 1e-12))
        var = float((w * (r - mean) ** 2).sum() / max(w_sum, 1e-12))

        rows.append({
            "state": k,
            "posterior_weight": float(w_sum),
            "daily_mean_return": mean,
            "daily_vol": sqrt(var),
            "annualised_mean_return": mean * config.annualisation,
            "annualised_vol": sqrt(var) * sqrt(config.annualisation)
        })

    return pd.DataFrame(rows)


train_state_stats_unsorted = weighted_state_return_stats(train_df, hmm_fit["posterior"], config.n_states)

state_order = train_state_stats_unsorted.sort_values("annualised_vol")["state"].to_list()
state_rank_map = {old_state: new_state for new_state, old_state in enumerate(state_order)}

train_state_stats_unsorted

In [ ]:
def reorder_hmm_fit(fit, state_order):
    order = np.array(state_order, dtype=int)

    reordered = {
        "pi": fit["pi"][order],
        "A": fit["A"][np.ix_(order, order)],
        "means": fit["means"][order],
        "covariances": fit["covariances"][order],
        "posterior": fit["posterior"][:, order],
        "loglik_history": fit["loglik_history"],
        "log_likelihood": fit["log_likelihood"],
        "n_iter": fit["n_iter"]
    }

    return reordered


hmm_sorted = reorder_hmm_fit(hmm_fit, state_order)

train_posterior = hmm_sorted["posterior"]
train_inferred_state = train_posterior.argmax(axis=1)

train_state_stats = weighted_state_return_stats(train_df, train_posterior, config.n_states)
train_state_stats["regime_label"] = ["calm", "normal", "stress"]

train_state_stats

## 14. Estimated transition matrix

The transition matrix tells us regime persistence.

Expected duration of state $i$:

$$
\mathbb E[D_i] = \frac{1}{1-A_{ii}}
$$

if $A_{ii}<1$.

In [ ]:
transition_estimated = pd.DataFrame(
    hmm_sorted["A"],
    columns=[f"to_{label}" for label in ["calm", "normal", "stress"]],
    index=[f"from_{label}" for label in ["calm", "normal", "stress"]]
)

transition_estimated

In [ ]:
duration_rows = []

for i, label in enumerate(["calm", "normal", "stress"]):
    persistence = hmm_sorted["A"][i, i]
    expected_duration = 1.0 / max(1e-12, 1.0 - persistence)
    duration_rows.append({
        "regime": label,
        "self_transition_probability": persistence,
        "expected_duration_days": expected_duration
    })

duration_report = pd.DataFrame(duration_rows)

duration_report

## 15. Viterbi algorithm

The posterior state:

$$
\arg\max_k p(Z_t=k\mid X_{1:T})
$$

gives the most likely state at each time individually.

The Viterbi algorithm gives the single most likely complete state path:

$$
\arg\max_{Z_{1:T}} p(Z_{1:T}\mid X_{1:T})
$$

This is useful for regime labelling.

In [ ]:
def viterbi_log(log_emissions, pi, A):
    T, K = log_emissions.shape

    log_pi = np.log(np.maximum(pi, 1e-300))
    log_A = np.log(np.maximum(A, 1e-300))

    delta = np.zeros((T, K))
    psi = np.zeros((T, K), dtype=int)

    delta[0] = log_pi + log_emissions[0]

    for t in range(1, T):
        scores = delta[t - 1][:, None] + log_A
        psi[t] = np.argmax(scores, axis=0)
        delta[t] = log_emissions[t] + np.max(scores, axis=0)

    states = np.zeros(T, dtype=int)
    states[-1] = np.argmax(delta[-1])

    for t in range(T - 2, -1, -1):
        states[t] = psi[t + 1, states[t + 1]]

    return states


train_log_emissions_sorted = gaussian_logpdf_full(
    X_train,
    hmm_sorted["means"],
    hmm_sorted["covariances"]
)

train_viterbi_state = viterbi_log(
    train_log_emissions_sorted,
    hmm_sorted["pi"],
    hmm_sorted["A"]
)

pd.Series(train_viterbi_state).value_counts().sort_index()

## 16. Training regime diagnostics

We compare:

- true state;
- posterior most likely state;
- Viterbi path.

Since labels are sorted by volatility, we should expect broad alignment, but not perfect one-to-one classification.

In [ ]:
train_regimes = train_df[["timestamp", "return", "price", "true_state"]].copy()
train_regimes["posterior_state"] = train_inferred_state
train_regimes["viterbi_state"] = train_viterbi_state

for k, label in enumerate(["calm", "normal", "stress"]):
    train_regimes[f"posterior_{label}"] = train_posterior[:, k]

train_regimes.head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(train_regimes["timestamp"], train_regimes["price"])
axes[0].set_title("Training Price")

axes[1].plot(train_regimes["timestamp"], train_regimes["true_state"], drawstyle="steps-post")
axes[1].set_title("True State")

axes[2].plot(train_regimes["timestamp"], train_regimes["viterbi_state"], drawstyle="steps-post")
axes[2].set_title("Viterbi-Inferred State")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
for label in ["calm", "normal", "stress"]:
    plt.plot(train_regimes["timestamp"], train_regimes[f"posterior_{label}"], label=label)
plt.title("Training Posterior Regime Probabilities")
plt.xlabel("Date")
plt.ylabel("Posterior probability")
plt.legend()
plt.show()

## 17. Test-period filtering

In live usage, future observations are not available.

Full-sample smoothing uses all observations, including future data.

For out-of-sample use, we need filtering:

$$
p(Z_t \mid X_{1:t})
$$

We recursively update filtered probabilities using only data up to time $t$.

In [ ]:
def filter_hmm_online(X, pi, A, means, covariances):
    log_emissions = gaussian_logpdf_full(X, means, covariances)
    log_A = np.log(np.maximum(A, 1e-300))

    T, K = log_emissions.shape

    filtered = np.zeros((T, K))
    log_alpha = np.zeros((T, K))

    log_alpha[0] = np.log(np.maximum(pi, 1e-300)) + log_emissions[0]
    log_alpha[0] -= logsumexp(log_alpha[0])

    filtered[0] = np.exp(log_alpha[0])

    for t in range(1, T):
        prediction = logsumexp(log_alpha[t - 1][:, None] + log_A, axis=0)
        log_alpha[t] = prediction + log_emissions[t]
        log_alpha[t] -= logsumexp(log_alpha[t])
        filtered[t] = np.exp(log_alpha[t])

    return filtered


# Start test filtering from the final filtered training distribution.
last_train_filtered = train_posterior[-1]

test_filtered = filter_hmm_online(
    X_test,
    pi=last_train_filtered,
    A=hmm_sorted["A"],
    means=hmm_sorted["means"],
    covariances=hmm_sorted["covariances"]
)

test_state = test_filtered.argmax(axis=1)

test_regimes = test_df[["timestamp", "return", "price", "true_state"]].copy()
test_regimes["filtered_state"] = test_state

for k, label in enumerate(["calm", "normal", "stress"]):
    test_regimes[f"prob_{label}"] = test_filtered[:, k]

test_regimes.head()

In [ ]:
plt.figure(figsize=(12, 6))
for label in ["calm", "normal", "stress"]:
    plt.plot(test_regimes["timestamp"], test_regimes[f"prob_{label}"], label=label)
plt.title("Out-of-Sample Filtered Regime Probabilities")
plt.xlabel("Date")
plt.ylabel("Filtered probability")
plt.legend()
plt.show()

## 18. Regime-conditioned return statistics

We compute test-period realised return statistics conditional on inferred regime.

This is the practical test:

> Do inferred regimes actually correspond to different risk/return behaviour?

In [ ]:
def regime_realised_stats(df, state_col, labels):
    rows = []

    for k, label in enumerate(labels):
        subset = df[df[state_col] == k]

        if len(subset) == 0:
            rows.append({
                "regime": label,
                "n_obs": 0,
                "daily_mean": np.nan,
                "daily_vol": np.nan,
                "annualised_mean": np.nan,
                "annualised_vol": np.nan,
                "hit_rate_positive": np.nan,
                "max_drawdown_proxy": np.nan
            })
            continue

        r = subset["return"]
        cum = (1 + r).cumprod()
        drawdown = cum / cum.cummax() - 1

        rows.append({
            "regime": label,
            "n_obs": len(subset),
            "daily_mean": r.mean(),
            "daily_vol": r.std(ddof=1),
            "annualised_mean": r.mean() * config.annualisation,
            "annualised_vol": r.std(ddof=1) * np.sqrt(config.annualisation),
            "hit_rate_positive": (r > 0).mean(),
            "max_drawdown_proxy": drawdown.min()
        })

    return pd.DataFrame(rows)


test_regime_stats = regime_realised_stats(test_regimes, "filtered_state", ["calm", "normal", "stress"])

test_regime_stats

## 19. Regime-conditioned volatility forecast

A simple regime-conditioned next-day volatility forecast is:

$$
\widehat{\sigma}_{t+1}^2 = \sum_k p_t(k)\sigma_k^2
$$

where:

$$
p_t(k)=p(Z_t=k\mid X_{1:t})
$$

and $\sigma_k^2$ is the estimated variance in regime $k$.

This is not as smooth as GARCH or HAR-RV, but it gives interpretable state-based risk estimates.

In [ ]:
state_daily_vars = train_state_stats["daily_vol"].to_numpy() ** 2

test_regimes["regime_variance_forecast"] = test_filtered @ state_daily_vars
test_regimes["regime_ann_vol_forecast"] = np.sqrt(test_regimes["regime_variance_forecast"] * config.annualisation)

test_regimes["realized_abs_return"] = test_regimes["return"].abs()
test_regimes["realized_proxy_ann_vol"] = test_regimes["realized_abs_return"] * np.sqrt(config.annualisation)

test_regimes[["timestamp", "regime_ann_vol_forecast", "realized_proxy_ann_vol"]].head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(test_regimes["timestamp"], test_regimes["regime_ann_vol_forecast"], label="Regime vol forecast", linewidth=2)
plt.plot(test_regimes["timestamp"], test_regimes["realized_proxy_ann_vol"], label="Abs return annualised proxy", alpha=0.35)
plt.title("Regime-Conditioned Volatility Forecast")
plt.xlabel("Date")
plt.ylabel("Annualised volatility")
plt.legend()
plt.show()

## 20. Toy regime-filtered strategy

A basic use case is risk filtering.

Suppose a simple trend/risk-on strategy holds the asset unless stress probability is high.

Define:

$$
w_t =
\begin{cases}
1, & p_t(\text{stress}) < 0.5 \\
0, & p_t(\text{stress}) \geq 0.5
\end{cases}
$$

This is not an alpha claim. It demonstrates how regime probabilities can control exposure.

In [ ]:
def toy_regime_filtered_strategy(test_regimes, stress_threshold=0.50):
    out = test_regimes.copy()

    out["weight"] = (out["prob_stress"] < stress_threshold).astype(float)
    out["strategy_return"] = out["weight"].shift(1).fillna(1.0) * out["return"]
    out["buy_hold_return"] = out["return"]

    out["cum_strategy"] = (1 + out["strategy_return"]).cumprod()
    out["cum_buy_hold"] = (1 + out["buy_hold_return"]).cumprod()

    return out


strategy_df = toy_regime_filtered_strategy(test_regimes, stress_threshold=0.50)

strategy_summary = pd.Series({
    "strategy_annualised_return": strategy_df["strategy_return"].mean() * config.annualisation,
    "buy_hold_annualised_return": strategy_df["buy_hold_return"].mean() * config.annualisation,
    "strategy_annualised_vol": strategy_df["strategy_return"].std(ddof=1) * np.sqrt(config.annualisation),
    "buy_hold_annualised_vol": strategy_df["buy_hold_return"].std(ddof=1) * np.sqrt(config.annualisation),
    "strategy_max_drawdown": (strategy_df["cum_strategy"] / strategy_df["cum_strategy"].cummax() - 1).min(),
    "buy_hold_max_drawdown": (strategy_df["cum_buy_hold"] / strategy_df["cum_buy_hold"].cummax() - 1).min(),
    "average_exposure": strategy_df["weight"].mean()
})

strategy_summary

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(strategy_df["timestamp"], strategy_df["cum_strategy"], label="Regime-filtered")
plt.plot(strategy_df["timestamp"], strategy_df["cum_buy_hold"], label="Buy and hold")
plt.title("Toy Regime-Filtered Strategy")
plt.xlabel("Date")
plt.ylabel("Cumulative growth")
plt.legend()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(strategy_df["timestamp"], strategy_df["weight"], drawstyle="steps-post")
plt.title("Regime Strategy Exposure")
plt.xlabel("Date")
plt.ylabel("Weight")
plt.ylim(-0.05, 1.05)
plt.show()

## 21. Confusion-style diagnostic

Because this is synthetic data, we can compare inferred states with true states.

Labels are not guaranteed to match perfectly, but after sorting by volatility, we expect broad alignment.

In real data this diagnostic is unavailable.

In [ ]:
def confusion_table(true_state, inferred_state):
    table = pd.crosstab(
        pd.Series(true_state, name="true_state"),
        pd.Series(inferred_state, name="inferred_state"),
        normalize="index"
    )
    return table


train_confusion = confusion_table(train_regimes["true_state"], train_regimes["viterbi_state"])
test_confusion = confusion_table(test_regimes["true_state"], test_regimes["filtered_state"])

train_confusion

In [ ]:
test_confusion

## 22. Practical checklist for regime models

Before trusting a regime-switching model, check:

1. **Feature design**  
   Are features available in real time, or do they leak future information?

2. **Scaling**  
   Are features standardised?

3. **Number of states**  
   Is $K$ economically interpretable?

4. **Initialisation**  
   Are results robust to initial parameters?

5. **Transition matrix**  
   Are regimes persistent but not absorbing?

6. **State interpretation**  
   Do states differ in return, volatility, skew, or drawdown?

7. **Filtered vs smoothed probabilities**  
   Are you using only past data for live decisions?

8. **Out-of-sample validation**  
   Do regimes predict realised risk?

9. **Strategy use**  
   Is the model used for risk control, sizing, or alpha?

10. **Stability**  
   Are parameters stable through time?

## 23. Saving outputs

The notebook saves:

1. synthetic regime-switching market data;
2. HMM feature table;
3. fitted HMM parameters;
4. transition matrix;
5. expected duration report;
6. training posterior probabilities;
7. test filtered probabilities;
8. regime statistics;
9. toy strategy output;
10. confusion diagnostics;
11. manifest.

In [ ]:
output_dir = Path("data/processed/markov_regime_switching_v1")
output_dir.mkdir(parents=True, exist_ok=True)

market_path = output_dir / "synthetic_regime_switching_market.csv"
features_path = output_dir / "hmm_features.csv"
transition_path = output_dir / "estimated_transition_matrix.csv"
duration_path = output_dir / "expected_duration_report.csv"
train_regimes_path = output_dir / "train_regime_probabilities.csv"
test_regimes_path = output_dir / "test_filtered_regime_probabilities.csv"
train_stats_path = output_dir / "train_state_statistics.csv"
test_stats_path = output_dir / "test_regime_statistics.csv"
strategy_path = output_dir / "toy_regime_filtered_strategy.csv"
strategy_summary_path = output_dir / "strategy_summary.csv"
train_confusion_path = output_dir / "train_confusion_table.csv"
test_confusion_path = output_dir / "test_confusion_table.csv"
params_path = output_dir / "hmm_parameters.json"
manifest_path = output_dir / "manifest.json"

market_df.to_csv(market_path, index=False)
hmm_df.to_csv(features_path, index=False)
transition_estimated.to_csv(transition_path)
duration_report.to_csv(duration_path, index=False)
train_regimes.to_csv(train_regimes_path, index=False)
test_regimes.to_csv(test_regimes_path, index=False)
train_state_stats.to_csv(train_stats_path, index=False)
test_regime_stats.to_csv(test_stats_path, index=False)
strategy_df.to_csv(strategy_path, index=False)
strategy_summary.to_frame("value").to_csv(strategy_summary_path)
train_confusion.to_csv(train_confusion_path)
test_confusion.to_csv(test_confusion_path)

params_payload = {
    "pi": hmm_sorted["pi"].tolist(),
    "A": hmm_sorted["A"].tolist(),
    "means": hmm_sorted["means"].tolist(),
    "covariances": hmm_sorted["covariances"].tolist(),
    "feature_cols": feature_cols,
    "regime_labels": ["calm", "normal", "stress"],
    "log_likelihood": float(hmm_sorted["log_likelihood"]),
    "n_iter": int(hmm_sorted["n_iter"])
}

params_path.write_text(json.dumps(params_payload, indent=2, default=str))

manifest = {
    "dataset_name": "markov_regime_switching_outputs",
    "schema_version": "markov_regime_switching_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "feature_cols": feature_cols,
    "regime_labels": ["calm", "normal", "stress"],
    "log_likelihood": float(hmm_sorted["log_likelihood"]),
    "n_iter": int(hmm_sorted["n_iter"]),
    "strategy_summary": strategy_summary.to_dict(),
    "notes": [
        "Synthetic data is generated from a three-state regime-switching return process.",
        "Gaussian HMM is implemented from scratch in log-space.",
        "Baum-Welch EM estimates transition probabilities and Gaussian emission parameters.",
        "States are sorted by inferred annualised volatility for interpretability.",
        "Test-period regime probabilities are filtered online using only past data.",
        "Toy regime-filtered strategy is a risk-control demonstration, not an alpha claim."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

market_path, transition_path, test_regimes_path, strategy_path, manifest_path

## 24. Limitations

### 24.1 Synthetic data

This notebook uses synthetic data with known hidden states.

Real markets do not reveal true regimes, so validation must rely on predictive usefulness and economic interpretation.

### 24.2 Gaussian emissions

The model assumes Gaussian emissions.

Financial returns can be heavy-tailed, skewed, and jumpy.

### 24.3 Fixed number of states

We choose $K=3$.

In practice, the number of states should be chosen using out-of-sample performance, interpretability, information criteria, or Bayesian methods.

### 24.4 Local optima

HMM likelihoods are non-convex.

Different initialisations can produce different regimes.

### 24.5 Regime labels are arbitrary

State labels must be sorted and interpreted after fitting.

### 24.6 Smoothed versus filtered probabilities

Smoothed probabilities use future information.

Live trading or risk systems must use filtered probabilities only.

### 24.7 Toy strategy is not alpha

The strategy section shows risk filtering only.

A real strategy requires transaction costs, slippage, robustness tests, and out-of-sample validation.

## 25. Research frontier and extensions

Important extensions include:

1. **Student-t HMM**  
   Heavy-tailed regime emissions.

2. **Gaussian mixture HMM**  
   More flexible within-regime distributions.

3. **Markov-switching GARCH**  
   Regime-dependent volatility dynamics.

4. **Hidden semi-Markov models**  
   Explicit duration modelling.

5. **Bayesian HMMs**  
   Posterior uncertainty over regimes and parameters.

6. **Online HMM learning**  
   Sequential parameter updates for live systems.

7. **Regime-switching factor models**  
   State-dependent factor returns and covariance matrices.

8. **Regime-aware portfolio construction**  
   Conditional volatility, correlation, and risk budgets by regime.

9. **Regime-switching CTA systems**  
   Trend-following exposure controlled by inferred market state.

10. **Chinese futures regime models**  
   Incorporate night sessions, price-limit locks, margin changes, and commodity-specific regimes.

## 26. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_04_vector_autoregression_granger_causality**  
   Modelling cross-asset lead-lag relationships.

2. **03_05_factor_momentum_and_volatility_scaling**  
   Combining trend and regime filters.

3. **03_06_cross_sectional_alpha_features**  
   Using regime probabilities as features.

4. **03_07_tree_models_for_return_prediction**  
   Nonlinear supervised learning with regime features.

5. **04_06_var_cvar_and_stress_testing**  
   Regime-conditioned risk forecasting.

6. **05_01_vectorized_backtest_engine**  
   Backtesting regime-filtered strategies with transaction costs.

7. **07_01_multi_asset_cta_research_pipeline**  
   Capstone CTA system with volatility and regime overlays.

## 27. Summary

This notebook implemented Markov regime switching with a Gaussian hidden Markov model.

It showed how to:

1. simulate regime-switching returns;
2. construct return and volatility features;
3. implement log-space forward-backward inference;
4. estimate HMM parameters with Baum-Welch EM;
5. sort regimes by volatility;
6. compute posterior probabilities;
7. infer the Viterbi path;
8. estimate transition probabilities and expected durations;
9. filter test-period regimes online;
10. compute regime-conditioned return statistics;
11. build a regime-conditioned volatility forecast;
12. demonstrate a toy regime-filtered strategy;
13. save outputs and diagnostics.

The key computational takeaway:

> HMMs require careful numerical implementation: log-space inference, stable covariance matrices, and robust initialisation.

The key financial takeaway:

> Regime probabilities turn market state into a quantitative feature for risk control, volatility forecasting, and strategy conditioning.

## 28. Further reading

- Hamilton, J. D. *Time Series Analysis.*
- Hamilton, J. D. "A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle."
- Rabiner, L. "A Tutorial on Hidden Markov Models and Selected Applications in Speech Recognition."
- Zucchini, MacDonald, and Langrock, *Hidden Markov Models for Time Series.*
- Literature on Markov-switching GARCH, Bayesian HMMs, and regime-switching portfolio allocation.